In [1]:
# ============================================================
# MODEL A — CLEAN TEXT (TITLE + ARTICLE)
# ============================================================

import pandas as pd
import numpy as np
import re
import joblib

from collections import Counter, defaultdict
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# =========================
# CONFIG
# =========================
DEV_PATH  = "../data/raw/development.csv"
EVAL_PATH = "../data/raw/evaluation.csv"

MODEL_PATH = "model_text_joint.joblib"
SUB_PATH   = "submission_text_joint.csv"

MIN_RULE_SUPPORT = 30
MIN_RULE_PURITY  = 0.95
C_VALUE = 1.5

# =========================
# LOAD + DROP TIMESTAMP
# =========================
df_dev  = pd.read_csv(DEV_PATH).drop(columns=["timestamp"], errors="ignore")
df_eval = pd.read_csv(EVAL_PATH).drop(columns=["timestamp"], errors="ignore")

# =========================
# CLEAN TEXT (HTML TAGS)
# =========================
HTML_RE = re.compile(r"<[^>]+>")

def clean_html(txt):
	if not isinstance(txt, str):
		return ""
	return HTML_RE.sub(" ", txt)

for df in (df_dev, df_eval):
	df["title"]   = df["title"].fillna("").astype(str).apply(clean_html)
	df["article"] = df["article"].fillna("").astype(str).apply(clean_html)
	df["source"]  = df["source"].fillna("").astype(str)

	df["text"] = (df["title"] + " " + df["article"]).str.lower()

# =========================
# NUMERIC
# =========================
def add_numeric(df):
	df["n_tokens"]    = df["article"].str.split().str.len()
	df["title_len"]   = df["title"].str.len()
	df["article_len"] = df["article"].str.len()
	df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)
	return df

df_dev  = add_numeric(df_dev)
df_eval = add_numeric(df_eval)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

for df in (df_dev, df_eval):
	df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

# =========================
# FEATURES
# =========================
FEATURES = ["source", "text"] + NUM_COLS

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int).values
X_eval = df_eval[FEATURES]

# =========================
# RULES
# =========================
def tokenize(text):
	return text.split()

def mine_rules(texts, labels):
	counts = defaultdict(lambda: Counter())
	for t, y in zip(texts, labels):
		for tok in set(tokenize(t)):
			counts[tok][int(y)] += 1

	rules = {}
	for tok, c in counts.items():
		total = sum(c.values())
		if total < MIN_RULE_SUPPORT:
			continue
		cls, freq = c.most_common(1)[0]
		if freq / total >= MIN_RULE_PURITY:
			rules[tok] = int(cls)
	return rules

def apply_rules(texts, rules):
	out = np.full(len(texts), -1)
	for i, t in enumerate(texts):
		for tok in set(tokenize(t)):
			if tok in rules:
				out[i] = rules[tok]
				break
	return out

# =========================
# MODEL
# =========================
model = Pipeline([
	("pre", ColumnTransformer([
		("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
		("w", TfidfVectorizer(ngram_range=(1,2), min_df=3, max_df=0.9, sublinear_tf=True), "text"),
		("c", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=3, max_df=0.9, sublinear_tf=True), "text"),
		("num", StandardScaler(), NUM_COLS)
	], n_jobs=-1)),
	("clf", LogisticRegression(C=C_VALUE, class_weight="balanced", max_iter=2000))
])

# =========================
# TRAIN + SAVE
# =========================
model.fit(X_dev, y_dev)
joblib.dump(model, MODEL_PATH)

rules = mine_rules(df_dev["text"], y_dev)

# =========================
# PREDICT + SUBMISSION
# =========================
pred = model.predict(X_eval)
rule_pred = apply_rules(df_eval["text"], rules)
pred[rule_pred != -1] = rule_pred[rule_pred != -1]

submission = pd.DataFrame({
	"Id": df_eval["Id"].values,
	"Predicted": pred.astype(int)
})

submission.to_csv(SUB_PATH, index=False)
print("Saved:", MODEL_PATH, SUB_PATH)


Saved: model_text_joint.joblib submission_text_joint.csv
